# Formative 3: Probability Distributions, Bayesian Probability and Gradient Descent

Mathematics for Machine Learning, African Leadership University. Group 11.

| Member | Leads | Module |
|---|---|---|
| Member 1: Milliam Mukamukiza | Part 1: EM algorithm | `src/em.py` |
| Member 2: Dedine Mukabucyana | Part 2: Bayesian probability | `src/bayes.py` |
| Member 3: Cedric Bigwi Hindura | Parts 3 and 4: gradient descent | `src/gradient_descent.py` |

All three members present every part. Algorithms live in `src/` and are imported here, so the notebook shows the results and the reasoning without duplicating code.

To run: `pip install -r requirements.txt`, place `IMDB Dataset.csv` in `data/` (see `data/README.md`), then Restart and Run All.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import em, bayes
from src import gradient_descent as gd
from src import plotting as pl
from src.data import load_galton

pl.use_plain_style()
pd.options.display.float_format = "{:.4f}".format

GALTON_CSV = "data/GaltonFamilies.csv"
IMDB_CSV = "data/IMDB Dataset.csv"
FIG = "figures"
os.makedirs(FIG, exist_ok=True)

---
# Part 1: Probability distributions and the EM algorithm

**Question.** Heights of two populations are pooled and the labels are deleted. Can a model discover the two groups on its own?

**Data.** Galton family heights. We compare **mothers and children** (one mother per family, every child). Set `PARENT = "father"` below to repeat the analysis with fathers. The labels are kept aside and used only to grade the result at the end.

In [ ]:
PARENT = "mother"          # "mother" or "father"
heights, labels, subgroup = load_galton(GALTON_CSV, PARENT)
print(f"{(labels == 0).sum()} {PARENT}s + {(labels == 1).sum()} children = {len(heights)} pooled heights")

### How EM works

Model: $p(x)=\pi_1\,\mathcal N(x\mid\mu_1,\sigma_1^2)+\pi_2\,\mathcal N(x\mid\mu_2,\sigma_2^2)$

| Step | What happens | Formula |
|---|---|---|
| Initialize | Means at the 25th and 75th percentiles, both variances equal to the data variance, $\pi_1=\pi_2=0.5$ | none |
| E-step | Every height gets a responsibility: the probability that it came from each component | $r_{ik}=\dfrac{\pi_k\,\mathcal N(x_i\mid\mu_k,\sigma_k^2)}{\sum_j \pi_j\,\mathcal N(x_i\mid\mu_j,\sigma_j^2)}$ |
| M-step | Re-estimate each component, weighting every height by its responsibility, with $N_k=\sum_i r_{ik}$ | $\mu_k=\dfrac{\sum_i r_{ik}x_i}{N_k},\quad \sigma_k^2=\dfrac{\sum_i r_{ik}(x_i-\mu_k)^2}{N_k},\quad \pi_k=\dfrac{N_k}{N}$ |
| Stop | The log-likelihood changes by less than $10^{-6}$ | $\lvert \ln L_t-\ln L_{t-1}\rvert<10^{-6}$ |

Implemented in `src/em.py`: `initialize`, `e_step`, `m_step`, `fit`.

### Run EM and show the required tracking table

In [ ]:
history = em.fit(heights)
final, final_ll = history[-1]
print(f"Stopped after {len(history) - 1} iterations, final log-likelihood {final_ll:.4f}\n")
em.tracking_table(history, n_iterations=2).style.hide(axis="index").format("{:.4f}", subset=pd.IndexSlice[:, "mu1":])

Component 1 is the lower-mean component and component 2 the higher-mean one. EM does not know which one a person would call mothers or children.

In [ ]:
pl.plot_em_snapshots(heights, history, path=f"{FIG}/em_snapshots.png"); plt.show()
pl.plot_loglik(history, path=f"{FIG}/em_loglik.png"); plt.show()

**Stopping rule.** The log-likelihood never decreases. It rises quickly in the first iterations and then flattens, so we stop when an iteration improves it by less than $10^{-6}$.

### Final fitted mixture

In [ ]:
print("means      :", final.means.round(4))
print("variances  :", final.variances.round(4))
print("proportions:", final.proportions.round(4))
pl.plot_mixture_fit(heights, final, "Two-component Gaussian mixture fitted by EM", path=f"{FIG}/em_fit.png"); plt.show()

### Should we simply cut at the global mean and average each pile?

**No.** Three reasons, shown below with the same data:

1. **Overlap.** A hard line labels everyone on the wrong side as wrong and cannot say "about 50/50". EM returns a probability for every height.
2. **Truncation.** Cutting at a point removes the tail of each group, so each pile's variance is too small and its mean is biased toward the cut.
3. **Forced sizes.** A cut at the mean forces two roughly equal piles. EM learns the means, variances and mixing proportions together by maximizing the likelihood.

In [ ]:
cut, split = em.global_mean_split(heights)
comparison = pd.DataFrame({
    "global-mean split": [*split.means, *split.variances, split.proportions[0], em.log_likelihood(heights, split)],
    "EM":                [*final.means, *final.variances, final.proportions[0], final_ll]},
    index=["mean 1", "mean 2", "variance 1", "variance 2", "proportion 1", "log-likelihood"])
display(comparison)
pl.plot_soft_vs_hard(heights, final, cut, path=f"{FIG}/soft_vs_hard.png"); plt.show()

### What did EM actually find?

If the two components were "parents" and "children", each sub-group would fall in one component. The table shows the average responsibility of component 2 (the taller one) for each sub-group.

In [ ]:
r = em.e_step(heights, final)
by_group = (pd.DataFrame({"group": subgroup, "height": heights, "P(component 2)": r[:, 1]})
            .groupby("group").agg(count=("height", "size"), mean_height=("height", "mean"),
                                  mean_P_component2=("P(component 2)", "mean")))
display(by_group)

EM separates **shorter from taller people**, which here mostly means **female from male**: the parent group and the daughters sit in the lower component and the sons in the upper one. EM is a density model, so it finds the dominant structure in the heights, not the label a human would choose. This explains the modest classification accuracy below.

In [ ]:
parent_component = em.closest_component(final, heights[labels == 0])   # component nearest the labelled parents' mean
scores = em.evaluate(heights, labels, final, parent_component)
print(f"Component {parent_component + 1} is treated as the {PARENT} group (its mean is closest to the {PARENT}s' mean).")
print(f"Accuracy {scores['accuracy']:.2%}   balanced accuracy {scores['balanced_accuracy']:.2%}   "
      f"always-guess-child baseline {scores['majority_baseline']:.2%}")
pl.plot_confusion(scores["confusion"], [PARENT, "child"], scores["accuracy"], path=f"{FIG}/em_confusion.png"); plt.show()

The evaluation reuses the data EM was fitted on, and accuracy is below the trivial baseline because 82% of the observations are children. We report this openly: the model is a valid mixture fit, but height alone barely identifies the label.

### Classify a test height

Change `TEST_HEIGHT` to the height given during the presentation and run the cell.

In [ ]:
TEST_HEIGHT = 68.0
p_parent, p_child = em.classify(TEST_HEIGHT, final, parent_component)
print(f"Height {TEST_HEIGHT} in:  P({PARENT}) = {p_parent:.4f}   P(child) = {p_child:.4f}   (sum = {p_parent + p_child:.4f})")
pl.plot_mixture_fit(heights, final, "Test height on the fitted mixture", test_height=TEST_HEIGHT); plt.show()

---
# Part 2: Bayesian probability on IMDb reviews

**Choice.** We compute $P(\text{Positive}\mid\text{keyword})$ only, not $P(\text{Negative}\mid\text{keyword})$.

**Data.** 50,000 IMDb reviews, half positive and half negative, so the prior is $P(\text{Positive})=0.5$.

**Method.** Basic Python only (`csv`, `re`, sets and counting). Each review is reduced to the set of words it contains.

| Quantity | Definition |
|---|---|
| Prior $P(\text{Positive})$ | positive reviews / all reviews |
| Likelihood $P(w\mid\text{Positive})$ | positive reviews containing $w$ / positive reviews |
| Marginal $P(w)$ | reviews containing $w$ / all reviews |
| Posterior $P(\text{Positive}\mid w)$ | likelihood $\times$ prior / marginal |

$$P(\text{Positive}\mid w)=\frac{P(w\mid\text{Positive})\,P(\text{Positive})}{P(w)}$$

**Keywords and why.**

- Positive: *excellent*, *wonderful*, *amazing* (direct praise) and *great* (very common, so it tests whether a popular word is also a reliable one).
- Negative: *terrible*, *awful* (direct criticism), *boring* (a common reason for a bad review) and *worst* (used in extreme complaints).

In [ ]:
POSITIVE = ["excellent", "wonderful", "amazing", "great"]
NEGATIVE = ["terrible", "awful", "boring", "worst"]
KEYWORDS = POSITIVE + NEGATIVE

reviews = bayes.load_reviews(IMDB_CSV)
print(len(reviews), "reviews loaded")
rows = bayes.bayes_table(reviews, KEYWORDS)
print(bayes.format_table(rows))

Worked example for one keyword, with the arithmetic written out:

In [ ]:
word, prior, likelihood, marginal, posterior = next(r for r in rows if r[0] == "worst")
print(f"P(Positive | {word}) = P({word} | Positive) x P(Positive) / P({word})")
print(f"                  = {likelihood:.4f} x {prior:.4f} / {marginal:.4f} = {posterior:.4f}")

In [ ]:
pl.plot_posteriors(KEYWORDS, [r[4] for r in rows], prior=rows[0][1], n_pos=len(POSITIVE), path=f"{FIG}/bayes_posteriors.png"); plt.show()

best_pos = max(rows[:len(POSITIVE)], key=lambda r: r[4]); weak_pos = min(rows[:len(POSITIVE)], key=lambda r: r[4])
best_neg = min(rows[len(POSITIVE):], key=lambda r: r[4])
print(f"Strongest positive signal: {best_pos[0]} ({best_pos[4]:.2f}).  Weakest positive signal: {weak_pos[0]} "
      f"({weak_pos[4]:.2f}, appears in {weak_pos[3]:.0%} of reviews).")
print(f"Strongest negative signal: {best_neg[0]} (only {best_neg[4]:.0%} of reviews containing it are positive).")
print("All positive keywords above the prior:", all(r[4] > r[1] for r in rows[:len(POSITIVE)]),
      "| all negative keywords below the prior:", all(r[4] < r[1] for r in rows[len(POSITIVE):]))

**Limits to mention.** A keyword is counted once per review (presence, not frequency). Negation is ignored ("not great" still counts as *great*). Each keyword is evaluated on its own, so nothing here assumes independence between words.

---
# Part 3: Gradient descent, manual calculation

**Model.** $\hat y = Xm + b$ with
$X=\begin{pmatrix}1&3\\4&10\end{pmatrix},\;
y=\begin{pmatrix}5\\6\end{pmatrix},\;
m_0=\begin{pmatrix}-1\\2\end{pmatrix},\;
b_0=\begin{pmatrix}1\\1\end{pmatrix}$

**Cost.** $J(m,b)=\dfrac1n(\hat y-y)^\top(\hat y-y)$ with $n=2$.

### Gradient by the chain rule

Let $u=\hat y-y=Xm+b-y$, so $J=\dfrac1n u^\top u$.

$$\frac{\partial J}{\partial u}=\frac{2}{n}u,\qquad \frac{\partial u}{\partial m}=X,\qquad \frac{\partial u}{\partial b}=I$$

$$\frac{\partial J}{\partial m}=\left(\frac{\partial u}{\partial m}\right)^{\!\top}\frac{\partial J}{\partial u}=\frac{2}{n}X^\top(\hat y-y)\qquad\qquad
\frac{\partial J}{\partial b}=\left(\frac{\partial u}{\partial b}\right)^{\!\top}\frac{\partial J}{\partial u}=\frac{2}{n}(\hat y-y)$$

**Update rule.** $m\leftarrow m-\alpha\,\dfrac{\partial J}{\partial m},\qquad b\leftarrow b-\alpha\,\dfrac{\partial J}{\partial b}$

**Assumptions we state explicitly.**

- $b=[1,1]^\top$ is a vector with one entry per data point, as given, so every operation is a matrix operation and nothing is treated as a scalar.
- The assignment does not give a learning rate; we use $\alpha=0.01$. The largest eigenvalue of the Hessian is about 127, so gradient descent is stable only for $\alpha<2/127\approx0.0157$.
- Number of updates = number of group members = 3. Each member does one by hand (see `OWNERS`).

In [ ]:
N_MEMBERS = 3
OWNERS = ["Milliam", "Dedine", "Cedric"]       # who performs each hand-calculated update
LR = 0.01
X  = np.array([[1., 3.], [4., 10.]])
y  = np.array([5., 6.])
m0 = np.array([-1., 2.])
b0 = np.array([1., 1.])

m_end, b_end, hist3 = gd.run_gradient_descent(X, y, m0, b0, LR, N_MEMBERS, use_scipy=False)
gd.update_table(hist3, N_MEMBERS, OWNERS).set_index("Update").T

The table is the answer key for the handwritten sheet (`docs/Part3_Manual_Calculations.pdf`): each column is one update with its predictions, errors, cost, both gradients and the new parameters.

**Trend.** The cost falls after every update: 61 → 6.5033 → 2.4974 → 2.1645. The parameters move against the gradient, so yes, they are moving in the direction that reduces the error. The first step is large because the initial gradient is large ($\partial J/\partial m=[45,\,113]$), and later steps shrink as the gradient flattens. $m_1$ becomes more negative, $m_2$ falls from 2 toward 1.1, $b_1$ rises slightly and $b_2$ falls slightly.

---
# Part 4: Gradient descent in code with SciPy

### 4.1 A function that takes an equation and returns its derivative

`gd.partial_derivative(f, theta, i)` accepts any equation `f` of a parameter vector and uses `scipy.differentiate.derivative` to compute $\partial f/\partial\theta_i$. `gd.scipy_gradient` stacks one derivative per parameter. First a check on a function whose derivative we know, $f(x,y)=x^3+2xy^2$:

In [ ]:
f = lambda t: t[0] ** 3 + 2 * t[0] * t[1] ** 2
theta = np.array([2.0, 3.0])
print("SciPy gradient :", np.round(gd.scipy_gradient(f, theta), 6))
print("Exact gradient :", [3 * 2**2 + 2 * 3**2, 4 * 2 * 3], " (3x^2 + 2y^2, 4xy)")

Now the same check on our MSE cost against the chain-rule formula from Part 3:

In [ ]:
g_scipy = gd.scipy_gradient(lambda t: gd.cost_of_theta(t, X, y), gd.pack(m0, b0))
dm_a, db_a = gd.analytic_gradients(X, m0, b0, y)
print("SciPy    [dJ/dm, dJ/db]:", np.round(g_scipy, 6))
print("Analytic [dJ/dm, dJ/db]:", np.round(np.concatenate([dm_a, db_a]), 6))
assert np.allclose(g_scipy, np.concatenate([dm_a, db_a]), atol=1e-6)

### 4.2 Every step visible

The loop below spells out each quantity: predictions, error, cost, the matrix product $X^\top u$, the SciPy gradient, and the updates. It runs for three updates so the numbers can be compared with the hand calculations. With $n=2$ the factor $2/n$ equals 1, so $X^\top u$ and $\partial J/\partial m$ print the same numbers.

In [ ]:
m, b = m0.copy(), b0.copy()
for t in range(1, N_MEMBERS + 1):
    y_hat = X @ m + b                                   # predictions
    u = y_hat - y                                       # error
    cost = u @ u / len(y)                               # MSE
    Xt_u = X.T @ u                                      # matrix product used by dJ/dm
    grad = gd.scipy_gradient(lambda th: gd.cost_of_theta(th, X, y), gd.pack(m, b))   # SciPy derivatives
    dm, db = grad[:2], grad[2:]
    m_new, b_new = m - LR * dm, b - LR * db             # updates
    print(f"Update {t} ({OWNERS[t - 1]})")
    print(f"  y_hat = {np.round(y_hat, 4)}   u = {np.round(u, 4)}   J = {cost:.4f}")
    print(f"  X^T u = {np.round(Xt_u, 4)}   dJ/dm = (2/n) X^T u = {np.round(dm, 4)}   dJ/db = (2/n) u = {np.round(db, 4)}")
    print(f"  m: {np.round(m, 4)} -> {np.round(m_new, 4)}     b: {np.round(b, 4)} -> {np.round(b_new, 4)}")
    m, b = m_new, b_new

### 4.3 Run to near convergence and plot

In [ ]:
N_ITER = 300
m_fin, b_fin, hist = gd.run_gradient_descent(X, y, m0, b0, LR, N_ITER)
its, M, B, J = gd.history_arrays(hist)

pl.plot_lines(its, {"m1": M[:, 0], "m2": M[:, 1], "b1": B[:, 0], "b2": B[:, 1]},
              "Parameters m and b over iterations", "iteration", "value", path=f"{FIG}/gd_parameters.png"); plt.show()
pl.plot_lines(its, {"MSE J(m, b)": J}, "Error over iterations (log scale)", "iteration", "MSE",
              path=f"{FIG}/gd_error.png", logy=True); plt.show()

The same two plots over the first 20 iterations, with the three hand-calculated updates marked, show the large early steps:

In [ ]:
k = 21
pl.plot_lines(its[:k], {"m1": M[:k, 0], "m2": M[:k, 1], "b1": B[:k, 0], "b2": B[:k, 1]},
              "Parameters, first 20 iterations", "iteration", "value", vlines={"update 3": N_MEMBERS}); plt.show()
pl.plot_lines(its[:k], {"MSE J(m, b)": J[:k]}, "Error, first 20 iterations", "iteration", "MSE",
              vlines={"update 3": N_MEMBERS}); plt.show()

### 4.4 Predictions with the final m and b

In [ ]:
y_pred = gd.predict(X, m_fin, b_fin)
print("final m =", np.round(m_fin, 4), "   final b =", np.round(b_fin, 4))
print(f"MSE: {J[0]:.1f} at the start, {J[3]:.4f} after the 3 hand-calculated updates, {J[-1]:.6f} after {N_ITER} iterations")
display(pd.DataFrame({"x": [tuple(row) for row in X], "y (true)": y, "y_hat (final)": y_pred, "error": y_pred - y}))
pl.plot_predictions(y, [(f"iteration {t}", hist[t]["y_hat"]) for t in (0, 3, len(hist) - 1)]); plt.show()

**Reading the result.** The error keeps falling toward zero. The parameters do not settle on unique values: $[X\;I]$ has rank 2 but there are 4 unknowns (two entries of $m$, two of $b$), so many $(m,b)$ fit the two data points exactly. The values reached depend on the starting point and the learning rate, which is why $b$ keeps drifting while the error is already tiny.

---
# Questions to be ready for

- **Why not split at the global mean?** Hard labels at an overlapping boundary, truncated piles with underestimated variances, forced equal sizes, and no uncertainty. See the Part 1 comparison table.
- **Why is the EM accuracy low?** EM recovers the dominant structure (shorter versus taller, roughly female versus male), not mother versus child. Accuracy is below the 82% always-guess-child baseline.
- **How do the components get names?** Component order is arbitrary; we name the component whose mean is closest to the labelled parents' mean. The posterior values are unaffected.
- **When does EM stop, and can it get stuck?** When the log-likelihood gain is below $10^{-6}$. It reaches a local maximum, so the initialization (percentiles here) matters.
- **Why $P(\text{Positive}\mid w)$ and not both?** The assignment asks for one. $P(\text{Negative}\mid w)=1-P(\text{Positive}\mid w)$ would add no information.
- **Why $\alpha=0.01$?** The brief gives none; stability requires $\alpha<2/\lambda_{\max}\approx0.0157$.
- **Why is $b$ a vector?** It is given as $[1,1]$, one entry per data point, so the update is a matrix operation rather than a scalar one.
- **Why do $m$ and $b$ not converge to unique values?** Four unknowns, two equations (rank 2).